# SPAR: Training Pipeline (Colab)

**SPAR** (Sensor-Policy with Adaptive Resource allocation) is a safe reinforcement learning framework that treats sensor selection as part of the policy. At each timestep the agent chooses *which* observation modalities to activate alongside the environment action, subject to a per-step sensor-cost budget enforced as a Constrained Markov Decision Process (CMDP). The Lagrangian multiplier λ adapts online to keep cumulative cost at or below the budget. This notebook trains SPAR on `highway-env` (no MuJoCo required) and visualises reward, cost, and per-sensor activation rates.

In [ ]:
# GPU detection
import torch
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

# Virtual display (guard for highway-env headless rendering)
import subprocess
subprocess.run(['apt-get', 'install', '-y', '-q', 'xvfb', 'python3-opengl'], check=False)
!pip install -q pyvirtualdisplay
from pyvirtualdisplay import Display
Display(visible=0, size=(1400, 900)).start()

In [ ]:
REPO_URL  = "https://github.com/OfekGlick/SPAR.git"
REPO_ROOT = "/content/SPAR"

import os
if not os.path.exists(REPO_ROOT):
    os.system(f"git clone {REPO_URL} {REPO_ROOT}")
os.chdir(REPO_ROOT)

# Pin numpy to 1.x before anything else — avoids ABI mismatch with Colab's pre-built packages
os.system("pip install -q 'numpy<2.0'")

# Install dependencies: highway-env (editable), then requirements, then root package
os.system(f"pip install -q -e {REPO_ROOT}/base_envs/highway_env")
os.system(f"pip install -q -r {REPO_ROOT}/requirements.txt")
os.system(f"pip install -q -e {REPO_ROOT}")
print("Install complete — RESTART RUNTIME now (Runtime > Restart runtime), then continue.")

> **Important:** After the cell above finishes, go to **Runtime → Restart runtime**, then continue running from the next cell. This ensures newly installed packages are available.

In [ ]:
import sys, os
REPO_ROOT = "/content/SPAR"
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)

import spar_envs.budget_aware_highway   # registers env IDs with OmniSafe
import omnisafe, torch, pandas as pd, matplotlib.pyplot as plt
from configs.highway_config import CUSTOM_CFGS
from utils.training_utils import train_agent_base
print("Imports OK")

In [ ]:
import copy

ENV_ID           = "budget-aware-highway-fast-v0"
ALGO             = "PPOLag"      # PPOLag = safe; PPO = unconstrained baseline
BUDGET           = 2.0           # sensor cost limit per step (max = 4 sensors active)
TOTAL_STEPS      = 49152         # ~6 epochs; change to 409600 for full training
SEED             = 42

custom_cfgs = copy.deepcopy(CUSTOM_CFGS)
custom_cfgs['logger_cfgs']['use_wandb']        = False
custom_cfgs['logger_cfgs']['use_tensorboard']  = False
custom_cfgs['logger_cfgs']['log_dir']          = '/content/runs'
custom_cfgs['train_cfgs']['total_steps']       = TOTAL_STEPS
custom_cfgs['train_cfgs']['device']            = 'cuda:0' if torch.cuda.is_available() else 'cpu'
custom_cfgs['algo_cfgs']['steps_per_epoch']    = 8192
custom_cfgs['env_cfgs']['max_episode_steps']   = 250
custom_cfgs['env_cfgs']['seed']                = SEED
custom_cfgs['env_cfgs']['use_all_obs']         = False  # False = SPAR; True = all-sensors baseline
custom_cfgs['algo_cfgs']['use_cost']           = True
custom_cfgs['lagrange_cfgs']                   = {'cost_limit': BUDGET}
custom_cfgs['algo_cfgs']['sd_regulizer']       = False
custom_cfgs['model_cfgs']['actor_type']        = 'auto'

print(f"Will run {TOTAL_STEPS // 8192} epochs ({TOTAL_STEPS} steps) on {custom_cfgs['train_cfgs']['device']}")

In [ ]:
import time
t0 = time.time()

train_agent_base(ALGO, ENV_ID, custom_cfgs, eval_num_episodes=10)

print(f"Done in {(time.time()-t0)/60:.1f} min")

In [ ]:
import glob

progress_files = sorted(glob.glob('/content/runs/**/*.csv', recursive=True), key=os.path.getmtime)
assert progress_files, "No progress.csv found — check training completed without errors"
progress_csv = progress_files[-1]
print("Progress file:", progress_csv)

df = pd.read_csv(progress_csv)
print(f"{len(df)} epochs logged")
print(df[['TotalEnvSteps', 'Metrics/EpRet', 'Metrics/EpCost']].tail())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# EpRet
axes[0].plot(df['TotalEnvSteps'], df['Metrics/EpRet'], '#2E86AB', lw=2)
axes[0].set(title='Episode Return', xlabel='Steps', ylabel='EpRet'); axes[0].grid(alpha=0.3)

# EpCost + budget line
axes[1].plot(df['TotalEnvSteps'], df['Metrics/EpCost'], '#C73E1D', lw=2, label='EpCost')
axes[1].axhline(BUDGET * 250, color='k', ls='--', label=f'Budget={BUDGET*250:.0f}')
axes[1].set(title='Episode Cost', xlabel='Steps', ylabel='EpCost'); axes[1].legend(); axes[1].grid(alpha=0.3)

# Lagrange multiplier (PPOLag only)
lag_col = next((c for c in df.columns if 'LagrangeMultiplier' in c and '/' not in c.replace('Metrics/LagrangeMultiplier','')), None)
if lag_col:
    axes[2].plot(df['TotalEnvSteps'], df[lag_col], '#6A994E', lw=2)
    axes[2].set(title='Lagrange Multiplier \u03bb', xlabel='Steps'); axes[2].grid(alpha=0.3)
else:
    axes[2].text(0.5, 0.5, 'N/A\n(PPO baseline)', ha='center', va='center', transform=axes[2].transAxes)
    axes[2].set_title('Lagrange Multiplier \u03bb')

plt.suptitle(f'{ALGO} on {ENV_ID} \u2014 budget={BUDGET}', fontweight='bold')
plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
sensor_cols = sorted([c for c in df.columns if 'EpActivationSensor' in c])
modality_names = ['Kinematics', 'LidarObservation', 'OccupancyGrid', 'TimeToCollision']
colors = ['#2E86AB', '#F18F01', '#C73E1D', '#6A994E']

if sensor_cols:
    fig, ax = plt.subplots(figsize=(9, 4))
    for i, col in enumerate(sensor_cols):
        idx = int(col.split('_')[-1])
        ax.plot(df['TotalEnvSteps'], df[col], color=colors[i], lw=2,
                label=modality_names[idx] if idx < 4 else f"Sensor_{idx}")
    ax.axhline(BUDGET / len(sensor_cols), color='k', ls='--', lw=1, label='Uniform budget')
    ax.set(ylim=(0, 1.05), title='Per-Sensor Activation Rate', xlabel='Steps', ylabel='Rate')
    ax.legend(fontsize=10); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('/content/sensor_activations.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("\nFinal epoch activation rates:")
    for i, col in enumerate(sensor_cols):
        print(f"  {modality_names[int(col.split('_')[-1])]}: {df[col].iloc[-1]:.3f}")
else:
    print("Sensor activation columns not found (expected when use_all_obs=True)")

## Scaling Up

Full-scale experiment (single run):
```bash
python run_spar_highway.py --algo PPOLag --env-id budget-aware-highway-fast-v0 \
    --budget 2.0 --total-steps 409600 --seed 42
```

All environments + budgets on a cluster (generates Slurm jobs from `configs/highway_config.py`):
```bash
python launch_highway.py --submit
```

Robosuite (requires MuJoCo ≥ 3.3.0):
```bash
cd base_envs/robosuite && pip install -e .
python run_spar_robosuite.py --algo PPOLag --env-id budget-aware-Lift --budget 3.0
```